# Introduction

----------------------------------------------------------------
## “Date Cleaning & Seasonality Analysis"
----------------------------------------------------------------

### Business Case: Okinsurance — Water-Sport Risk Insurance for US Hotels & Resorts
#### This notebook focuses on cleaning and analyzing the country, activity and type column of the GSAF dataset, in order to test 
#### Hypothesis 1 "Shark-incident risk is concentrated in small number of countries" and Hypothesis 3 "unprovoked attacks dominate and  represent environmental risk".
#### Our analysis focuses on three core dimensions: Country, activity and attack type.
#### Date and Month cleaning are handled in a separate notebook and merged here for the country + month cross-analysis.
#### Our analysis focuses on three core dimensions: 
#### Country, Activity, and Attack Type, with a dedicated section for swimming‑specific risk, which directly supports the business question. 
-----------------------------------------------------------------------------------------------------------------------------------------------------

# Shark Attacks Data Cleaning
### 1. Load dataset
### 2. Explore Structure
### 3. Check duplicates
### 4. Check missing values
### 5. Clean columns
### 6. Standardize text
### 7. Handle missing values
### 8. Analyze Country, Type and Activity

-------------------------------
##### Dataset:
----------------------------------

#### Global Shark Attack File (GSAF) — https://www.sharkattackfile.net/spreadsheets/GSAF5.xls

#### Importing Libraries Pandas & Regex 

In [ ]:
import re
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
from cleaning import clean_country, clean_type, clean_activity
from visuals import (
    plot_top_countries,
    plot_attack_types,
    plot_top_activities,
    plot_swimming_by_country,
    plot_swim_vs_surf_country,
    plot_swim_vs_surf_month
)


-------------------------------------------------------------------

## 1. Load Data 

In [ ]:
df = pd.read_excel("https://www.sharkattackfile.net/spreadsheets/GSAF5.xls", engine="xlrd")

In [ ]:
df.head()

In [ ]:
df.tail()

## 2. Explore Structure: First look at the data

In [ ]:
df.columns.tolist()

### Quick inspection : Before cleaning 

In [ ]:
df.info()  

### Checking Columns : Country, Activity and Type 

In [ ]:
df[["Country", "Activity", "Type"]].info()

##### Finding: 
##### We have 7103 entries = 7103 Rows
##### Data Type (Country, Activity, Type) = str (object) type


-------------------------------------------------
## 3. Checking Duplicates

In [ ]:
duplicates_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates_count}")

duplicate_rows = df[df.duplicated()]
duplicate_rows.head(20)

##### Findings : No Duplicate Found :) 

------------------
## 4. Missing values for the Country, Activity and Type 

In [ ]:
df[["Type", "Country", "Activity"]].isna().sum().to_frame("missing_count")

##### Activity has highest number of missing values (583)

---------------------
### Missing values overview before cleaning

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)


-----------------------------------------------
### Checking for Unique values nunique ()
------------------------------------------------

#### Unique values and spread before cleaning

In [ ]:
df["Type"].nunique(dropna=True), df["Country"].nunique(dropna=True), df["Activity"].nunique(dropna=True)


##### Finding:
##### Type = 14 (low)
##### Country = 254 (High) -> may indicate inconsistent formatting
##### Activity = 1613 (too high) -> highly inconsistent, too many variant 

------------------------------
## Value counts inspection
------------------------------

### Value counts for Country

In [ ]:
df["Country"].value_counts(dropna=False).to_frame("count").head(20)

##### Findings:
##### Multiple formats : "AUSTRALIA" "Australia" 
##### there is also ? : "Sudan?" "RED SEA?" "Africa"
-----------------------------------------------------


### Value counts for Activity

In [ ]:
df["Activity"].value_counts(dropna=False).to_frame("count").head(20)

##### Findings:
##### Many variations for a specific activity: Swimming appears twice
##### "Fishing" and "Spearfishing", "Scuba diving", "Free diving", "Pearl diving"

------------------------------------------------------------------
### Value counts for Type

In [ ]:
df["Type"].value_counts(dropna=False).to_frame("count").head(20)

##### Findings: 
##### Duplicate spelling variations : "Unprovoked"; "UNprovoked" ; "unprovoked"
##### Invalid input : "?" 
---------------------------------------------------------------------------------

------------------------------------

# 5. Cleaning Process 

------------------------------------------------------------------------
### Cleaning Plan for df2

##### Create a working copy of the dataset to protect the original file.
##### Standardize text format with consistent case (lower or title case)
##### Remove punctuation or extra spaces.
##### Convert `?` with "Unknown" or "NaN".
##### Merge values ex: "Unprovoked", "Swimming"
##### Activity regrouping or category
##### Recheck missing values after cleaning.

-------------------------------------------------------------
##### Create a working copy
##### To ensure the original dataset remains unchanged
##### All cleaning steps will be applied to this copy
##### So that we can also run test before and after

In [ ]:
df2 = df.copy()

#### Confirming if the copy exist

In [ ]:
df2 = df.copy()
df2.head()

In [ ]:
df2 = df.copy()
df2.info()

------------------------------------------------
#### Formatting Standardization
##### All country names were converted to title case to ensure consistent capitalization (e.g., “AUSTRALIA” → “Australia”).
##### Question marks and punctuation were removed to eliminate uncertainty markers (e.g., “Sudan?” → “Sudan”).
##### This step reduces artificial variation caused by formatting differences.

#### Convert all values to title case and remove punctuation

In [ ]:
df2["Country"] = (
    df2["Country"]
    .str.strip()
    .str.title()               
    .str.replace(r"\?", "", regex=True)  
    .str.replace(r"[^\w\s]", "", regex=True)  
)


---------------------------------
#### Identify non-country values
---------------------------------

In [ ]:
df2["Country"].value_counts().head(20)

--------------------------------------------------------
#### Option 1 " Our Analyis is focus on the Top 20 "
##### Replace with Unknown : "Africa", "Asia", "Red Sea"
---------------------------------------------------------

In [ ]:
region_list = ["Africa", "Asia", "Red Sea"]
df2["Country"] = df2["Country"].replace(region_list, "Unknown")

In [ ]:
df2["Country"].value_counts()

In [ ]:
df2[df2["Country"] == "Unknown"].head()


###### Findings : " Between " is not an actual country but can be use to describe a region
###### Used regex to replace any 'Between X And Y' style entries with 'Unknown'
###### Just for clarity and fun :) 

In [ ]:
df2["Country"] = df2["Country"].replace(r"^Between.*", "Unknown", regex=True)

In [ ]:
df2["Country"].value_counts().head(20)

--------------------------------------------------------
### Plan for Cleaning 'Type' column
--------------------------------------------------------
##### Converting all values to lowercase for consistency.
##### Removing punctuation and question marks (e.g., "?" → "").
##### Strip whitespace "  provoked"
##### Replace missing or unclear values with "Unknown".
##### Creating categories using a mapping dictionary.
---------------------------------------------------------

In [ ]:
df2["Type"] = df2["Type"].str.lower()
df2["Type"] = df2["Type"].str.replace(r"[^\w\s]", "", regex=True)
df2["Type"] = df2["Type"].str.strip()  
df2["Type"] = df2["Type"].replace("", "Unknown")
df2["Type"] = df2["Type"].fillna("Unknown")

type_map = {
    "unprovoked": "Unprovoked",
    "provoked": "Provoked",
    "invalid": "Invalid",
    "watercraft": "Watercraft",
    "sea disaster": "Sea Disaster",
    "questionable": "Questionable",
    "unconfirmed": "Unconfirmed",
    "unverified": "Unverified",
    "under investigation": "Under Investigation",
    "boat": "Boat"
}

df2["Type"] = df2["Type"].replace(type_map)

df2["Type"].value_counts()


----------------------------------------------
### Rare group Type categories into "Other"
-----------------------------------------------

In [ ]:
top_types = df2["Type"].value_counts().nlargest(5).index
df2.loc[~df2["Type"].isin(top_types), "Type"] = "Other"


In [ ]:
df2["Type"].value_counts()

---------------------------
### Display "Other" 
---------------------------

In [ ]:
df2[df2["Type"] == "Other"].head(20)

#### How many rows became "Other" 

In [ ]:
df2["Type"].value_counts().loc["Other"]


##### Sanity Check :) 

In [ ]:
top_types


--------------------------------------------
###  Plan for Cleaning 'Activity' column
-------------------------------------------
##### Converting all in Lowercase
##### Remove punctuation (?, ., /, -, etc.)
##### Strip whitespace
##### Replace missing values
##### Fix common typos
##### Map messy descriptions to core categories
##### Group rare activities into "Other"
-------------------------------------------------

In [ ]:
df2["Activity"] = df2["Activity"].str.lower()

df2["Activity"] = df2["Activity"].str.replace(r"[^\w\s]", " ", regex=True)

df2["Activity"] = df2["Activity"].str.strip()

df2["Activity"] = df2["Activity"].replace("", "unknown")
df2["Activity"] = df2["Activity"].fillna("unknown")

typo_map = {
    "swmming": "swimming",
    "swimmingq": "swimming",
    "surf sking": "surf skiing",
    "surf skiing": "surfing",
    "surf ski": "surfing",
    "surf skiing": "surfing"
}
df2["Activity"] = df2["Activity"].replace(typo_map)

activity_map = {
    "surfing": "surfing",
    "surf": "surfing",
    "surf bathing": "surfing",
    "surf paddling": "surfing",
    "surf skiing": "surfing",
    "surf fishing": "fishing",
    "surf fishing wading": "fishing",
    "swimming": "swimming",
    "treading water": "swimming",
    "standing": "standing",
    "wading": "wading",
    "diving": "diving",
    "free diving": "diving",
    "scuba diving": "diving",
    "snorkeling": "snorkeling",
    "spearfishing": "spearfishing",
    "fishing": "fishing",
    "fell overboard": "fell overboard",
    "kayaking": "kayaking",
    "bathing": "bathing",
    "body boarding": "body boarding",
    "body surfing": "body surfing",
    "pearl diving": "diving"
}

df2["Activity"] = df2["Activity"].replace(activity_map)

#### Grouping rare activities into "Other" 

In [ ]:
top_activities = df2["Activity"].value_counts().nlargest(20).index
df2.loc[~df2["Activity"].isin(top_activities), "Activity"] = "other"

#### Just checking for confirmation :) 

In [ ]:
df2["Activity"].value_counts().head(20)

#####################
#       EDA SETUP   # 
#####################

-------------------------
## Cleaned DataFrame ##
---------------------------

In [ ]:
df2.head()

## Basic structure of the dataset ##

In [ ]:
df2.info()

## Quick Overview 
#### Numerical vs Categorical 
##### Dataset size and main fields. 

In [ ]:
df2.describe(include="all")

------------------------------------
#### Top 10 countries by incident count
------------------------------------

In [ ]:
country_counts = df2["Country"].value_counts().head(10)
country_percent = country_counts / country_counts.sum() * 100

country_summary = pd.DataFrame({
    "count": country_counts,
    "percent": country_percent.round(1)
})
country_summary


-----------------------------
#### Bar chart: Top 10 countries
-----------------------------

In [ ]:
plt.figure()
sns.barplot(
    x=country_summary.index,
    y=country_summary["percent"],
    palette="Blues_d"
)
plt.title("Top 10 Countries by Shark Incident Share (%)")
plt.ylabel("Percentage of incidents (%)")
plt.xlabel("Country")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### Note: USA, Australia, South Africa dominate incident 

-----------------------------
#### Attack Type distribution
-----------------------------

In [ ]:
type_counts = df2["Type"].value_counts()
type_percent = type_counts / type_counts.sum() * 100

type_summary = pd.DataFrame({
    "count": type_counts,
    "percent": type_percent.round(1)
})
type_summary

-----------------------------
#### Bar chart: Attack Types
----------------------------

In [ ]:
plt.figure()
sns.barplot(
    x=type_summary.index,
    y=type_summary["percent"],
    palette="Reds_d"
)
plt.title("Distribution of Attack Types (%)")
plt.ylabel("Percentage of incidents (%)")
plt.xlabel("Type")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### Findings
##### Unprovoked vs Provoked vs Invalid
##### 74% Unprovoked, 9.1% Provoked, 7.8% Invalid.”

-----------------------------
#### Top 20 activities
-----------------------------

In [ ]:
activity_counts = df2["Activity"].value_counts().head(20)
activity_percent = activity_counts / activity_counts.sum() * 100

activity_summary = pd.DataFrame({
    "count": activity_counts,
    "percent": activity_percent.round(1)
})
activity_summary


-----------------------------
#### Bar chart: Top Activities
----------------------------

In [ ]:
plt.figure()
sns.barplot(
    x=activity_summary.index,
    y=activity_summary["percent"],
    palette="Greens_d"
)
plt.title("Top 20 Activities by Incident Share (%)")
plt.ylabel("Percentage of incidents (%)")
plt.xlabel("Activity")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

##### Note:  Surfing, Swimming, Fishing, Diving as dominant activities  

-----------------------------
### Swimming-only subset
-----------------------------


In [ ]:
swim_df = df2[df2["Activity"] == "swimming"]
swim_df.shape


---------------------------------
#### Swimming incidents by country
--------------------------------

In [ ]:
swim_country_counts = swim_df["Country"].value_counts().head(10)
swim_country_percent = swim_country_counts / swim_country_counts.sum() * 100

swim_country_summary = pd.DataFrame({
    "count": swim_country_counts,
    "percent": swim_country_percent.round(1)
})
swim_country_summary

In [ ]:
plt.figure()
sns.barplot(
    x=swim_country_summary.index,
    y=swim_country_summary["percent"],
    palette="Blues"
)
plt.title("Top 10 Countries for Swimming-Related Incidents (%)")
plt.ylabel("Percentage of swimming incidents (%)")
plt.xlabel("Country")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


##### Note: Top 3 countries are USA, Australia and South Africa